### **[Three Geopolitical Shocks. Three Completely Different Markets](https://medium.com/predict/a40b3eda1d42)**

In [ ]:
import os
import sys

import warnings
warnings.filterwarnings('ignore')

import json
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules

%autosave 20

In [ ]:
try:
  if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get('EODHD_API_KEY')
    USE_EODHD = True
  else:
    api_key = os.environ.get('EODHD_API_KEY')
    USE_EODHD = True
except:
  api_key = None
  USE_EODHD = False

#### **The Asset Basket and Data Source**

In [ ]:
events = {
  'oct7_attack': {
    'date': '2023-10-07',
    'label': 'Hamas Attack on Israel (Oct 2023)',
    'shock_type': 'confidence',
    'shock_label': 'Type 1 - Confidence Shock'
  },
  'yen_carry_unwind': {
    'date': '2024-08-05',
    'label': 'Yen Carry Unwind + Middle East Escalation (Aug 2024)',
    'shock_type': 'liquidity',
    'shock_label': 'Type 2 - Liquidity Shock'
  },
  'tariff_shock': {
    'date': '2025-04-03',
    'label': 'US-China Tariff Shock (Apr 2025)',
    'shock_type': 'structural',
    'shock_label': 'Type 3 - Structural Shock'
  }
}

assets = {
  'spy': 'SPY.US', 'qqq': 'QQQ.US', 'iwm': 'IWM.US',
  'xle': 'XLE.US', 'xlf': 'XLF.US', 'ita': 'ITA.US',
  'xlk': 'XLK.US', 'gld': 'GLD.US', 'tlt': 'TLT.US',
  'uup': 'UUP.US', 'vixy': 'VIXY.US'
}

def fetch_prices(ticker, start, end):
  url = f'https://eodhd.com/api/eod/{ticker}'
  params = {
    'from': start,
    'to': end,
    'api_token': api_key,
    'fmt': 'json'
  }
  r = requests.get(url, params=params)
  df = pd.DataFrame(r.json())
  df['date'] = pd.to_datetime(df['date'])
  df = df.set_index('date')[['adjusted_close']]
  df.columns = [ticker.split('.')[0].lower()]
  return df

def fetch_event_prices(event_date, lookback=30, lookahead=30):
  start = (pd.Timestamp(event_date) - pd.Timedelta(days=lookback)).strftime('%Y-%m-%d')
  end = (pd.Timestamp(event_date) + pd.Timedelta(days=lookahead)).strftime('%Y-%m-%d')
  frames = [fetch_prices(ticker, start, end) for ticker in assets.values()]
  return pd.concat(frames, axis=1)

# Fetch event prices
event_prices = {name: fetch_event_prices(e['date']) for name, e in events.items()}
event_prices.keys()

#### **The Repricing Sequence Engine**

In [ ]:
def normalize_to_event(df, event_date):
  event_date = pd.Timestamp(event_date)
  valid_dates = df.index[df.index >= event_date]
  anchor = valid_dates[0]
  normalized = df.div(df.loc[anchor]) * 100
  return normalized, anchor

def get_event_window(df, anchor, t_minus=5, t_plus=10):
  start_idx = df.index.get_loc(anchor) - t_minus
  end_idx = df.index.get_loc(anchor) + t_plus
  start_idx = max(start_idx, 0)
  return df.iloc[start_idx:end_idx + 1]

def repricing_leaderboard(window_df, anchor):
  anchor_idx = window_df.index.get_loc(anchor)
  post_event = window_df.iloc[anchor_idx:]
  cumulative_returns = (post_event / post_event.iloc[0] - 1) * 100
  t1_moves = cumulative_returns.iloc[1].abs().sort_values(ascending=False)
  return cumulative_returns, t1_moves


event_windows = {}
leaderboards = {}

for name, meta in events.items():
  df = event_prices[name]
  normalized, anchor = normalize_to_event(df, meta['date'])
  window = get_event_window(normalized, anchor)
  cumret, t1_rank = repricing_leaderboard(window, anchor)
  event_windows[name] = {'window': window, 'anchor': anchor, 'cumret': cumret}
  leaderboards[name] = t1_rank
  print(f"\n{meta['label']}")
  print(f'anchor date: {anchor.date()}')
  print('T+1 move ranking:')
  print(t1_rank.round(2))

#### **Options Data and IV Skew**

In [ ]:
def fetch_options_all(ticker, start, end, exp_cap):
  url = 'https://eodhd.com/api/mp/unicornbay/options/eod'
  all_records = []
  offset = 0
  limit = 1000
  cols = None

  while True:
    params = {
      'filter[underlying_symbol]': ticker,
      'filter[tradetime_from]': start,
      'filter[tradetime_to]': end,
      'filter[exp_date_to]': exp_cap,
      'fields[options-eod]': 'type,exp_date,strike,volatility,tradetime',
      'page[limit]': limit,
      'page[offset]': offset,
      'api_token': api_key,
      'compact': 1
    }
    r = requests.get(url, params=params)
    payload = r.json()

    if 'meta' not in payload:
      print(f'unexpected response at offset {offset}: {list(payload.keys())}')
      break

    if cols is None:
      cols = [f.strip() for f in payload['meta']['fields']]

    batch = payload['data']
    all_records.extend(batch)

    total = payload['meta']['total']
    offset += limit
    if offset >= total or not batch:
      break

  df = pd.DataFrame(all_records, columns=cols)
  df['tradetime'] = pd.to_datetime(df['tradetime'])
  df['exp_date'] = pd.to_datetime(df['exp_date'])
  df['strike'] = pd.to_numeric(df['strike'], errors='coerce')
  df['volatility'] = pd.to_numeric(df['volatility'], errors='coerce')
  return df.dropna(subset=['volatility', 'strike']).query('volatility > 0')

def compute_skew(df, spot):
  df = df.copy()
  df['moneyness'] = df['strike'] / spot

  for expiry in sorted(df['exp_date'].unique()):
    sub = df[df['exp_date'] == expiry]
    otm_puts = sub[(sub['type'] == 'put') & (sub['moneyness'].between(0.90, 0.97))]
    atm_calls = sub[(sub['type'] == 'call') & (sub['moneyness'].between(0.97, 1.03))]
    if otm_puts.empty or atm_calls.empty:
      continue

    daily_skew = []
    for date, puts in otm_puts.groupby('tradetime'):
      calls = atm_calls[atm_calls['tradetime'] == date]
      if calls.empty:
        continue
      skew = puts['volatility'].mean() - calls['volatility'].mean()
      daily_skew.append({'date': date, 'skew': skew})

    if daily_skew:
      print(f'  using expiry: {expiry.date()}, {len(daily_skew)} days')
      return pd.DataFrame(daily_skew).set_index('date').sort_index()

  return pd.DataFrame()


spy_skew = {}

for name, meta in events.items():
  anchor = event_windows[name]['anchor']
  spot = event_prices[name].loc[anchor, 'spy']
  start = (anchor - pd.Timedelta(days=20)).strftime('%Y-%m-%d')
  end = (anchor + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
  exp_cap = (pd.Timestamp(end) + pd.Timedelta(days=90)).strftime('%Y-%m-%d')
  raw = fetch_options_all('SPY', start, end, exp_cap)
  print(f'\n{meta["label"]} | total rows: {len(raw)}')
  skew_df = compute_skew(raw, spot)
  spy_skew[name] = skew_df
  print(skew_df)

#### **Composite Stress Score**

In [ ]:
def build_composite(event_name, skew_df, event_prices_df, anchor):
  prices = event_prices_df[['spy', 'gld']].copy()
  prices['corr'] = prices['spy'].rolling(10).corr(prices['gld'])

  def zscore(s):
    return (s - s.mean()) / s.std()

  skew_z = zscore(skew_df['skew'])
  corr_z = zscore(prices['corr'].dropna())

  corr_z = corr_z * -1

  combined = pd.concat([skew_z.rename('skew_z'), corr_z.rename('corr_z')], axis=1).dropna()
  combined['composite'] = combined.mean(axis=1)

  combined['stress_flag'] = combined['composite'] > 1.0

  return combined


composites = {}
for name, meta in events.items():
  anchor = event_windows[name]['anchor']
  skew_df = spy_skew[name]
  prices_df = event_prices[name]
  comp = build_composite(name, skew_df, prices_df, anchor)
  composites[name] = comp
  print(f"\n{meta['label']}")
  print(comp.round(3))

#### **News Sentiment**

In [ ]:
def fetch_sentiment(ticker, start, end):
  url = 'https://eodhd.com/api/sentiments'
  params = {
    's': ticker,
    'from': start,
    'to': end,
    'api_token': api_key,
    'fmt': 'json'
  }
  r = requests.get(url, params=params)
  data = r.json()
  key = ticker if ticker in data else ticker + '.US'
  if key not in data:
    return pd.DataFrame()
  df = pd.DataFrame(data[key])
  df['date'] = pd.to_datetime(df['date'])
  df = df.set_index('date')[['normalized']].rename(columns={'normalized': 'sentiment'})
  return df.sort_index()


event_sentiment = {}
for name, meta in events.items():
  anchor = event_windows[name]['anchor']
  start = (anchor - pd.Timedelta(days=20)).strftime('%Y-%m-%d')
  end = (anchor + pd.Timedelta(days=10)).strftime('%Y-%m-%d')
  sent_df = fetch_sentiment('SPY', start, end)
  event_sentiment[name] = sent_df
  print(f"\n{meta['label']}")
  print(sent_df)

#### **Event 1: Hamas Attack on Israel, Oct 7 2023**

#### **Event 2: Yen Carry Unwind, Aug 5 2024**

#### **Event 3: US-China Tariff Shock, Apr 2025**

#### **Putting It All Together: The Heatmap**

In [ ]:
fig = make_subplots(rows=1, cols=3,
  subplot_titles=[e['label'] for e in events.values()],
  horizontal_spacing=0.08)

for i, (name, meta) in enumerate(events.items()):
  window = event_windows[name]['window']
  anchor = event_windows[name]['anchor']
  anchor_idx = window.index.get_loc(anchor)

  start_i = max(anchor_idx - 3, 0)
  end_i = min(anchor_idx + 8, len(window))
  slice_df = window.iloc[start_i:end_i].copy()
  slice_df.columns = [c.upper() for c in slice_df.columns]

  anchor_pos = anchor_idx - start_i
  anchor_vals = slice_df.iloc[anchor_pos]
  pct_df = ((slice_df - anchor_vals) / anchor_vals * 100).round(2)

  n_days = len(pct_df)
  t_labels = [f'T{d:+d}' for d in range(-anchor_pos, -anchor_pos + n_days)]

  fig.add_trace(go.Heatmap(
    z=pct_df.values.T,
    x=t_labels,
    y=list(pct_df.columns),
    colorscale='RdYlGn',
    zmid=0,
    zmin=-15,
    zmax=15,
    showscale=(i == 2),
    colorbar=dict(title='% return from T0')
  ), row=1, col=i+1)

fig.update_layout(
  title='Asset Return Heatmap - T-3 to T+7 across Events',
  template='plotly_dark',
  height=500
)

for annotation in fig['layout']['annotations']:
  annotation['font'] = dict(size=11)
  annotation['y'] = 1.02

fig.show()